# BPE Tokenizer

Notebook for exploring and testing the byte-pair encoding tokenizer work for CS336 Assignment 1.

## Setup

Run this notebook from the repository root so local package imports and test paths resolve correctly.

In [10]:
from pathlib import Path

REPO_ROOT = Path.cwd()
REPO_ROOT

PosixPath('/home/ngejay3046/cs336-assignment-1')

## Data Inspection

Use this section to inspect small text samples before training or debugging the tokenizer.

In [11]:
sample_text = "low lower lowest wider widest"
sample_text

'low lower lowest wider widest'

## Tokenizer Experiments

Import your tokenizer implementation here once it is available through the assignment adapter or package modules.

In [12]:
# Example placeholder:
# from cs336_basics.tokenizer import BPETokenizer
# tokenizer = BPETokenizer(...)
# tokenizer.encode(sample_text)

## Focused Tests

Run tokenizer-specific tests from a terminal with:

```sh
uv run pytest tests/test_tokenizer.py
```

## Assignment Integration

### ord() and chr() Functions

In Python, you can use the ord() function to convert a single Unicode character into its integer representation. The chr() function converts an integer Unicode code point into a string with the corresponding character.

In [ ]:
from IPython.display import display

display(ord('牛'))
display(chr(29275))

29275

'牛'

### Problem (unicode1): Understanding Unicode

(a) What Unicode character does `chr(0)` return?

(b) How does this character’s string representation differ from its printed representation?

(c) What happens when this character occurs in text?

**Answer.**

(a) `chr(0)` returns the Unicode character with code point `U+0000`, commonly called **NUL** or the **null character**. In Python's escaped representation, this appears as `'\x00'`.

(b) Its string representation shows an escaped form because the character itself is not visually printable. For example, `repr(chr(0))` returns `"'\\x00'"`. Printing it with `print(chr(0))` writes the actual null character to the output stream, but it appears invisible.

(c) When `U+0000` occurs in text, it usually has no visible glyph. Python allows it inside strings and treats it as an ordinary character of length 1. However, it can still cause issues in systems or formats that treat NUL specially. For example, older C-style string APIs historically use the null byte as a string terminator, so embedded NUL characters may affect how text is processed outside Python.

In [ ]:
x = chr(0)
display(x)
display(repr(chr(0)))
print("chr(0):", x)
print("length:", len(x))
display("this is a test" + chr(0) + "string")
print("this is a test" + chr(0) + "string")

'\x00'

"'\\x00'"

chr(0):  
length: 1


'this is a test\x00string'

this is a test string


### Unicode Encodings

To encode a Unicode string into UTF-8, we can use the encode() function in Python. To access the underlying byte values for a Python bytes object, we can iterate over it (e.g., call list()). Finally, we can use the decode() function to decode a UTF-8 byte string into a Unicode string.

In [ ]:
test_string = "hello! こんにちは!"

# Encode the string into UTF-8 bytes.
utf8_encoded = test_string.encode("utf-8")

# Display the encoded byte string.
print("Encoded byte string:", utf8_encoded)

# The type of the encoded string is 'bytes', which is a sequence of byte values (integers from 0 to 255).
print("Type of encoded byte string:", type(utf8_encoded))

# Get the list of byte values for the encoded string (a list of integers from 0 to 255).
print("Byte values:", list(utf8_encoded))

# One byte does not necessarily correspond to one Unicode character!
print("Length of original string:", len(test_string))

# The length of the UTF-8 encoded byte string is different from the length of the original Unicode string, 
# because some characters (like 'こ', 'ん', 'に', 'ち', 'は') are represented by multiple bytes in UTF-8.
print("Length of UTF-8 encoded byte string:", len(utf8_encoded))

# Finally, we can decode the UTF-8 byte string back into a Unicode string.
print("Decoded string:", utf8_encoded.decode("utf-8"))

Encoded byte string: b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
Type of encoded byte string: <class 'bytes'>
Byte values: [104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
Length of original string: 13
Length of UTF-8 encoded byte string: 23
Decoded string: hello! こんにちは!


### Problem (unicode2): Unicode Encodings

(a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

(b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
```

> 'hello'

(c) Give a two-byte sequence that does not decode to any Unicode character(s).

**Answer.**

(a) UTF-8 is usually preferable for byte-level tokenizer training because it is compact for ASCII-heavy text, backwards-compatible with ASCII, and does not introduce many predictable zero bytes. For example, the ASCII character `a` is one byte in UTF-8 (`61`), but two bytes in UTF-16 (`61 00` in little-endian form) and four bytes in UTF-32 (`61 00 00 00`). Since much web and code text is ASCII-heavy, UTF-16 and UTF-32 would make the training corpus longer and force the tokenizer to model many artificial bytes that come from the encoding rather than from the text itself. UTF-8 also has no required byte-order mark and is the standard encoding for most modern text data, so training on UTF-8 bytes better matches the data distribution we usually want to tokenize.

(b) The function is incorrect because UTF-8 characters may require multiple bytes. It decodes each byte independently, so it only works for single-byte ASCII characters. For a multibyte character, the individual bytes are not valid standalone UTF-8 strings. For example, `"é".encode("utf-8")` is `b'\xc3\xa9'`; decoding `b'\xc3'` by itself raises a `UnicodeDecodeError` because it is only the first byte of a two-byte UTF-8 sequence. Similarly, `"こんにちは".encode("utf-8")` contains three-byte UTF-8 sequences, and decoding one byte at a time fails.

(c) One example is `b'\xc3\x28'`. The byte `0xc3` indicates the start of a two-byte UTF-8 sequence, but `0x28` is not a valid continuation byte because UTF-8 continuation bytes must be in the range `0x80` to `0xbf`.


### BPE Tokenizer Training

In this coding task, we will implement a (similar to) GPT-2 style UTF-8 byte-level BPE tokenizer training algorithm, in a manner that will pass the unit tests provided in the repository.

The BPE tokenizer training procedure that we will implement will consist of following steps:

#### 1. Vocabulary initialization

The tokenizer vocabulary is a one-to-one mapping from bytestring token to integer ID. Since we’re training a byte-level BPE tokenizer, our initial vocabulary is simply the set of all bytes. Since there are 256 possible byte values, our initial vocabulary is of size 256.

#### 2. Pre-tokenization

Most modern tokenizers use a regex-based pre-tokenizer, a practice from GPT-2. We’ll use a slightly enhanced form of the original regex, fetched from 
[github.com/openai/tiktoken/pull/234/files](https://github.com/openai/tiktoken/pull/234/files):

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

TIP: Use re.finditer to avoid storing the pre-tokenized words as you construct your mapping from pre-tokens to their counts. For example: 

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
import regex as re
def get_pre_tokens(text: str):
    return [m.group(0) for m in re.finditer(PAT, text)]

pre_tokens = get_pre_tokens("sample_text")
print(pre_tokens)
```

> ['sample', '_', 'text']

#### 3. Compute BPE merges

Now that we’ve converted our input text into pre-tokens and represented each pre-token as a sequence of UTF-8 bytes, we can compute the BPE merges (i.e., train the BPE tokenizer). 

At a high level, the BPE algorithm iteratively counts every pair of bytes and identifies the pair (“A”, “B”) with the highest frequency. Every occurrence of this most frequent pair (“A”, “B”) is then merged, i.e., replaced with a new token “AB”. This new merged token is added to our vocabulary; as a result, the size of the final vocabulary after BPE training is the size of the initial vocabulary (256 in our case), plus the number of BPE merge operations performed during training. 

NOTE: For efficiency during BPE training, we do not consider pairs that cross pre-token boundaries.

When computing merges, **deterministically break ties in pair frequency by preferring the lexicographically greater pair**. For example, if the pairs (“A”, “B”), (“A”, “C”), (“B”, “ZZ”), and (“BA”, “A”) all have the highest frequency, we’d merge (“BA”, “A”):

> Merge one of [("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")] by lexicographic ordering, assuming all of the pairs have the same highest counts.

> Result: Merge ('BA', 'A')

#### 4. Handling Special Tokens

We will always use `<|endoftext|>` as a special token. 

All special tokens should always be preserved as a single token (i.e., a single integer ID). These special tokens must be added to the vocabulary, so they have a corresponding fixed token ID.

#### 5. Optimize the BPE training algorithm

For a start, we will be dealing with relatively small text corpora, but memory usage must remain controlled. Optimize the BPE training algorithm to be **as time-efficient as possible**, subject to reasonable memory usage. 

A major bottleneck is the pre-tokenization step. One possible method is to speed up pre-tokenization by parallelizing your code with the built-in library `multiprocessing`. Concretely, in parallel implementations of pre-tokenization, we can chunk the corpus while ensuring that chunk boundaries occur at the beginning of a special token (for the purpose of this coding task, the special token will be `<|endoftext|>`). This chunking will always be valid, since we never want to merge across document boundaries. The edge case of receiving a very large corpus that does not contain `<|endoftext|>` can be safely ignored for the purpose of this coding task.

#### 6. Removing special tokens before pre-tokenization

Before running pre-tokenization with the regex pattern (using `re.finditer`), you should strip out all special tokens from your corpus (or your chunk, if using a parallel implementation). Make sure that you split on your special tokens, so that no merging can occur across the text they delimit. For example, if you have a corpus (or chunk) like `[Doc 1]<|endoftext|>[Doc 2]`, you should split on the special token `<|endoftext|>`, and pre-tokenize `[Doc 1]` and `[Doc 2]` separately, so that no merging can occur across the document boundary. In other words, special tokens define hard segmentation boundaries during training, **but they should not themselves contribute to merge counts**. This can be done using `re.split` with `"|".join(special_tokens)` as the delimiter (with careful use of `re.escape` since `|` may occur in the special tokens).

#### 7. Optimizing the merging step

BPE training speed can be improved by indexing the counts of all pairs and incrementally updating these counts, rather than explicitly iterating over each pair of bytes to count pair frequencies. You can get significant speedups with this caching procedure, though we note that the merging part of BPE training is not parallelizable in Python.

#### 8. The coding task

Write a function that, given a path to an input text file, trains a (byte-level) BPE tokenizer. Your BPE training function must handle the following input parameters, but more is allowed.

**Input**

input_path: `str` 
- Path to a text file with BPE tokenizer training data.

vocab_size: `int` 
- A positive integer that defines the maximum final vocabulary size (including the initial byte vocabulary, vocabulary items produced from merging, and any special tokens).

special_tokens: `list[str]` 
- A list of strings to add to the vocabulary. During training, treat them as hard boundaries that prevent merges across their spans, but do not include them when computing merge statistics.

Your BPE training function should return the resulting vocabulary and merges:

**Output**

vocab: `dict[int, bytes]` 
- The tokenizer vocabulary, a mapping from int (token ID in the vocabulary) to bytes (token bytes).

merges: `list[tuple[bytes, bytes]]` 
- A list of BPE merges produced from training. Each list item is a tuple of bytes (`<token1>`, `<token2>`), representing that `<token1>` was merged with `<token2>`. The merges should be ordered by order of creation.

To test your BPE training function against our provided tests, you will first need to implement the test adapter at [adapters.run_train_bpe]. Then, run `uv run pytest tests/test_train_bpe.py`. Your implementation should be able to pass all tests.


In [ ]:
import heapq
import os
from collections import Counter, defaultdict
from dataclasses import dataclass

import regex as re


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
PRETOKEN_RE = re.compile(PAT)

Pair = tuple[bytes, bytes]
BYTE_TOKENS = tuple(bytes([i]) for i in range(256))


@dataclass(frozen=True)
class _ReversePair:
    pair: Pair

    def __lt__(self, other: "_ReversePair") -> bool:
        return self.pair > other.pair


def _pretoken_counts(text: str, special_tokens: list[str]) -> Counter[bytes]:
    counts: Counter[bytes] = Counter()
    if special_tokens:
        escaped_specials = [re.escape(token) for token in sorted(special_tokens, key=len, reverse=True)]
        chunks = re.split("|".join(escaped_specials), text)
    else:
        chunks = [text]

    for chunk in chunks:
        for match in PRETOKEN_RE.finditer(chunk):
            counts[match.group(0).encode("utf-8")] += 1
    return counts


def _word_pair_frequencies(word: list[bytes]) -> dict[Pair, int]:
    frequencies: dict[Pair, int] = {}
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        frequencies[pair] = frequencies.get(pair, 0) + 1
    return frequencies


def _merge_word(word: list[bytes], pair: Pair, merged_token: bytes) -> list[bytes]:
    merged_word: list[bytes] = []
    i = 0
    while i < len(word):
        if i + 1 < len(word) and word[i] == pair[0] and word[i + 1] == pair[1]:
            merged_word.append(merged_token)
            i += 2
        else:
            merged_word.append(word[i])
            i += 1
    return merged_word


def train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
) -> tuple[dict[int, bytes], list[Pair]]:
    with open(input_path, encoding="utf-8") as f:
        text = f.read()

    vocab: dict[int, bytes] = {i: BYTE_TOKENS[i] for i in range(256)}
    vocab_values = set(vocab.values())
    for special_token in special_tokens:
        special_bytes = special_token.encode("utf-8")
        if special_bytes not in vocab_values:
            vocab[len(vocab)] = special_bytes
            vocab_values.add(special_bytes)

    pretoken_counts = _pretoken_counts(text, special_tokens)
    word_counts: list[int] = []
    words: list[list[bytes]] = []
    for pretoken, count in pretoken_counts.items():
        word_counts.append(count)
        words.append([BYTE_TOKENS[byte] for byte in pretoken])

    pair_counts: dict[Pair, int] = defaultdict(int)
    pair_to_word_ids: dict[Pair, set[int]] = defaultdict(set)
    for word_id, word in enumerate(words):
        for pair, frequency in _word_pair_frequencies(word).items():
            pair_counts[pair] += frequency * word_counts[word_id]
            pair_to_word_ids[pair].add(word_id)

    heap: list[tuple[int, _ReversePair, Pair]] = []

    def push_pair(pair: Pair) -> None:
        count = pair_counts.get(pair, 0)
        if count > 0:
            heapq.heappush(heap, (-count, _ReversePair(pair), pair))

    for pair in pair_counts:
        push_pair(pair)

    def pop_best_pair() -> Pair | None:
        while heap:
            neg_count, _, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count > 0 and count == -neg_count:
                return pair
        return None

    merges: list[Pair] = []
    while len(vocab) < vocab_size:
        best_pair = pop_best_pair()
        if best_pair is None:
            break

        merged_token = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        vocab[len(vocab)] = merged_token

        affected_word_ids = list(pair_to_word_ids.get(best_pair, ()))
        changed_pairs: set[Pair] = set()
        for word_id in affected_word_ids:
            word_count = word_counts[word_id]
            old_word = words[word_id]
            old_pairs = _word_pair_frequencies(old_word)

            for pair, frequency in old_pairs.items():
                next_count = pair_counts[pair] - frequency * word_count
                if next_count > 0:
                    pair_counts[pair] = next_count
                else:
                    del pair_counts[pair]

                word_ids = pair_to_word_ids.get(pair)
                if word_ids is not None:
                    word_ids.discard(word_id)
                    if not word_ids:
                        del pair_to_word_ids[pair]
                changed_pairs.add(pair)

            new_word = _merge_word(old_word, best_pair, merged_token)
            words[word_id] = new_word
            new_pairs = _word_pair_frequencies(new_word)

            for pair, frequency in new_pairs.items():
                pair_counts[pair] += frequency * word_count
                pair_to_word_ids[pair].add(word_id)
                changed_pairs.add(pair)

        for pair in changed_pairs:
            push_pair(pair)

    return vocab, merges


### How the BPE training code works

The trainer first initializes the vocabulary with the 256 possible byte values, then appends any special tokens as complete UTF-8 byte strings. Special tokens are used as hard document boundaries during training: the corpus is split on them before regex pre-tokenization, so they never contribute to pair counts and no merge can cross their span.

After pre-tokenization, the code stores a `Counter` from each unique pre-token byte string to its corpus frequency. Each unique pre-token is represented as a list of byte-level token symbols. BPE training then repeatedly selects the most frequent adjacent pair, breaks frequency ties by choosing the lexicographically greatest byte pair, adds the merged token to the vocabulary, and rewrites all affected pre-token sequences left-to-right.

The implementation is time efficient because it avoids rescanning the whole corpus after every merge. It maintains `pair_counts`, a global table of weighted adjacent-pair frequencies, and `pair_to_word_ids`, an index from each pair to only the unique pre-token sequences that currently contain it. When a pair is merged, only those affected pre-token sequences are updated. Their old pair contributions are subtracted, the merge is applied, and their new pair contributions are added. A heap gives fast access to the next best pair, while lazy invalidation avoids expensive deletion or priority updates inside the heap.

For larger corpora, the main improvement would be parallel pre-tokenization. The file can be chunked at `<|endoftext|>` boundaries, each worker can build a local pre-token `Counter`, and the counters can be summed before the sequential BPE merge loop begins. Memory use can also be improved by storing token IDs instead of byte strings inside `words`, streaming file chunks instead of reading the whole file at once, and periodically rebuilding the heap if stale heap entries become too numerous. The merge loop itself is inherently sequential because each merge changes the statistics used to choose the next merge.


### Scaling BPE training to larger corpora

The baseline trainer above is a good correctness-first implementation: it counts unique pre-tokens, maintains pair frequencies incrementally, and avoids rescanning the full corpus after every merge. That is enough for the assignment fixtures, but a 3GB corpus changes the bottlenecks. Reading the full file into one Python string, running regex pre-tokenization in one process, storing every word token as a `bytes` object, and allowing the lazy heap to accumulate many stale entries can all become expensive.

The most useful scaling improvement is to parallelize the work that is independent before the merge loop starts. Pre-tokenization can be split across document boundaries such as `<|endoftext|>`, because BPE training should never merge across those boundaries anyway. Each process can count pre-tokens locally, and the parent process can reduce those counters before the usual sequential BPE merge loop begins. The merge loop remains sequential because each chosen merge changes the pair statistics used to choose the next merge.

The enhanced version below is therefore additive rather than a replacement for the baseline. It keeps the original implementation available for comparison, preserves deterministic output, and focuses first on time control through multiprocessing, then on memory control through lower-overhead integer token representations and heap rebuilding.


In [ ]:
from __future__ import annotations

import heapq
import json
import math
import multiprocessing as mp
import os
import pickle
import time
from collections import Counter, defaultdict
from collections.abc import Iterable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import BinaryIO

import regex as re


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
PRETOKEN_RE = re.compile(PAT)

Pair = tuple[bytes, bytes]
TokenPair = tuple[int, int]
BYTE_TOKENS = tuple(bytes([i]) for i in range(256))

_DEFAULT_CHUNK_BYTES = 64 * 1024 * 1024
_MIN_PARALLEL_BYTES = 16 * 1024 * 1024
_MIN_PARALLEL_WORDS = 20_000
_VOCAB_FILENAME = "vocab.pkl"
_MERGES_FILENAME = "merges.pkl"
_VOCAB_JSON_FILENAME = "vocab.json"
_MERGES_TEXT_FILENAME = "merges.txt"
_METADATA_FILENAME = "metadata.json"


@dataclass(frozen=True)
class _ReverseBytesPair:
    pair: Pair

    def __lt__(self, other: _ReverseBytesPair) -> bool:
        return self.pair > other.pair


def _multiprocessing_context() -> mp.context.BaseContext:
    try:
        return mp.get_context("fork")
    except ValueError:
        return mp.get_context()


def _resolve_num_workers(num_workers: int | None) -> int:
    if num_workers is None:
        return max(1, min(os.cpu_count() or 1, 8))
    if num_workers < 1:
        raise ValueError("num_workers must be at least 1")
    return num_workers


def _format_duration(elapsed_seconds: float) -> str:
    minutes = int(elapsed_seconds // 60)
    seconds = elapsed_seconds - minutes * 60
    return f"{minutes} min {seconds:.2f} sec"


def _format_duration_map(durations: dict[str, float]) -> dict[str, str]:
    return {name: _format_duration(duration) for name, duration in durations.items()}


def _default_output_dir(input_path: str | os.PathLike, vocab_size: int) -> Path:
    path = Path(input_path)
    return path.with_name(f"{path.stem}_bpe_{vocab_size}")


def _decode_utf8(token: bytes) -> str | None:
    try:
        return token.decode("utf-8")
    except UnicodeDecodeError:
        return None


def _vocab_json_entry(token_id: int, token: bytes) -> dict[str, object]:
    return {
        "id": token_id,
        "byte_values": list(token),
        "hex": token.hex(),
        "repr": repr(token),
        "utf8": _decode_utf8(token),
    }


def _write_vocab_json(vocab: dict[int, bytes], output_path: Path) -> None:
    payload = {
        "format": "cs336_basics.enhanced_bpe.v1",
        "tokens": [_vocab_json_entry(token_id, vocab[token_id]) for token_id in sorted(vocab)],
    }
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
        f.write("\n")


def _write_merges_text(merges: list[Pair], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as f:
        f.write("# cs336_basics enhanced BPE merges v1\n")
        f.write("# rank\tleft_repr\tright_repr\tmerged_repr\n")
        for rank, (left, right) in enumerate(merges):
            f.write(f"{rank}\t{left!r}\t{right!r}\t{left + right!r}\n")


def _write_training_metadata(metadata: dict[str, object], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
        f.write("\n")


def _write_training_artifacts(
    vocab: dict[int, bytes],
    merges: list[Pair],
    output_dir: str | os.PathLike | None,
    input_path: str | os.PathLike,
    vocab_size: int,
) -> Path:
    resolved_output_dir = Path(output_dir) if output_dir is not None else _default_output_dir(input_path, vocab_size)
    resolved_output_dir.mkdir(parents=True, exist_ok=True)

    with (resolved_output_dir / _VOCAB_FILENAME).open("wb") as f:
        pickle.dump(vocab, f, protocol=pickle.HIGHEST_PROTOCOL)

    with (resolved_output_dir / _MERGES_FILENAME).open("wb") as f:
        pickle.dump(merges, f, protocol=pickle.HIGHEST_PROTOCOL)

    _write_vocab_json(vocab, resolved_output_dir / _VOCAB_JSON_FILENAME)
    _write_merges_text(merges, resolved_output_dir / _MERGES_TEXT_FILENAME)
    return resolved_output_dir


def _pretoken_counts_from_text(text: str, special_tokens: list[str]) -> Counter[bytes]:
    counts: Counter[bytes] = Counter()
    if special_tokens:
        escaped_specials = [re.escape(token) for token in sorted(special_tokens, key=len, reverse=True)]
        chunks = re.split("|".join(escaped_specials), text)
    else:
        chunks = [text]

    for chunk in chunks:
        for match in PRETOKEN_RE.finditer(chunk):
            counts[match.group(0).encode("utf-8")] += 1
    return counts


def _pretoken_counts_for_range(args: tuple[str, int, int, list[str]]) -> Counter[bytes]:
    input_path, start, end, special_tokens = args
    with open(input_path, "rb") as f:
        f.seek(start)
        text = f.read(end - start).decode("utf-8")
    return _pretoken_counts_from_text(text, special_tokens)


def _find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)
    if file_size == 0:
        return [0]

    desired_num_chunks = max(1, min(desired_num_chunks, file_size))
    chunk_size = max(1, file_size // desired_num_chunks)
    chunk_boundaries = [min(i * chunk_size, file_size) for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096
    for boundary_index in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[boundary_index]
        file.seek(initial_position)
        while True:
            mini_chunk = file.read(mini_chunk_size)
            if mini_chunk == b"":
                chunk_boundaries[boundary_index] = file_size
                break

            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[boundary_index] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    return sorted(set(chunk_boundaries))


def _chunk_ranges(
    input_path: str,
    num_workers: int,
    chunk_bytes: int | None,
    special_tokens: list[str],
) -> list[tuple[int, int]]:
    file_size = os.path.getsize(input_path)
    if file_size == 0:
        return [(0, 0)]
    if not special_tokens:
        return [(0, file_size)]

    target_chunk_bytes = chunk_bytes or _DEFAULT_CHUNK_BYTES
    if target_chunk_bytes < 1:
        raise ValueError("chunk_bytes must be at least 1")

    desired_chunks = max(num_workers, math.ceil(file_size / target_chunk_bytes))
    with open(input_path, "rb") as f:
        boundaries = _find_chunk_boundaries(f, desired_chunks, special_tokens[0].encode("utf-8"))

    ranges = [(start, end) for start, end in zip(boundaries[:-1], boundaries[1:]) if end > start]
    return ranges or [(0, file_size)]


def _pretoken_counts(
    input_path: str | os.PathLike,
    special_tokens: list[str],
    num_workers: int,
    chunk_bytes: int | None,
) -> Counter[bytes]:
    path = os.fspath(input_path)
    file_size = os.path.getsize(path)
    if num_workers == 1 or file_size < _MIN_PARALLEL_BYTES or not special_tokens:
        with open(path, encoding="utf-8") as f:
            return _pretoken_counts_from_text(f.read(), special_tokens)

    ranges = _chunk_ranges(path, num_workers, chunk_bytes, special_tokens)
    if len(ranges) == 1:
        with open(path, encoding="utf-8") as f:
            return _pretoken_counts_from_text(f.read(), special_tokens)

    counts: Counter[bytes] = Counter()
    jobs = [(path, start, end, special_tokens) for start, end in ranges]
    context = _multiprocessing_context()
    worker_count = min(num_workers, len(jobs))
    with context.Pool(processes=worker_count) as pool:
        for partial_counts in pool.imap_unordered(_pretoken_counts_for_range, jobs, chunksize=1):
            counts.update(partial_counts)
    return counts


def _word_pair_frequencies(word: Sequence[int]) -> dict[TokenPair, int]:
    frequencies: dict[TokenPair, int] = {}
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        frequencies[pair] = frequencies.get(pair, 0) + 1
    return frequencies


def _merge_word(word: Sequence[int], pair: TokenPair, merged_token_id: int) -> list[int]:
    merged_word: list[int] = []
    i = 0
    while i < len(word):
        if i + 1 < len(word) and word[i] == pair[0] and word[i + 1] == pair[1]:
            merged_word.append(merged_token_id)
            i += 2
        else:
            merged_word.append(word[i])
            i += 1
    return merged_word


def _initial_pair_state_worker(
    records: list[tuple[int, list[int], int]],
) -> tuple[dict[TokenPair, int], dict[TokenPair, set[int]]]:
    pair_counts: dict[TokenPair, int] = {}
    pair_to_word_ids: dict[TokenPair, set[int]] = defaultdict(set)
    for word_id, word, word_count in records:
        for pair, frequency in _word_pair_frequencies(word).items():
            pair_counts[pair] = pair_counts.get(pair, 0) + frequency * word_count
            pair_to_word_ids[pair].add(word_id)
    return pair_counts, dict(pair_to_word_ids)


def _word_jobs(
    words: list[list[int]],
    word_counts: list[int],
    chunk_size: int,
) -> Iterable[list[tuple[int, list[int], int]]]:
    records: list[tuple[int, list[int], int]] = []
    for word_id, word in enumerate(words):
        records.append((word_id, word, word_counts[word_id]))
        if len(records) == chunk_size:
            yield records
            records = []
    if records:
        yield records


def _build_initial_pair_state(
    words: list[list[int]],
    word_counts: list[int],
    num_workers: int,
) -> tuple[dict[TokenPair, int], dict[TokenPair, set[int]]]:
    if num_workers == 1 or len(words) < _MIN_PARALLEL_WORDS:
        return _initial_pair_state_worker([(word_id, word, word_counts[word_id]) for word_id, word in enumerate(words)])

    pair_counts: dict[TokenPair, int] = defaultdict(int)
    pair_to_word_ids: dict[TokenPair, set[int]] = defaultdict(set)
    worker_count = min(num_workers, len(words))
    chunk_size = max(1, math.ceil(len(words) / (worker_count * 4)))
    context = _multiprocessing_context()

    with context.Pool(processes=worker_count) as pool:
        for local_pair_counts, local_pair_to_word_ids in pool.imap_unordered(
            _initial_pair_state_worker,
            _word_jobs(words, word_counts, chunk_size),
            chunksize=1,
        ):
            for pair, count in local_pair_counts.items():
                pair_counts[pair] += count
            for pair, word_ids in local_pair_to_word_ids.items():
                pair_to_word_ids[pair].update(word_ids)

    return dict(pair_counts), dict(pair_to_word_ids)


def train_bpe(
    input_path: str | os.PathLike,
    vocab_size: int,
    special_tokens: list[str],
    *,
    num_workers: int | None = None,
    chunk_bytes: int | None = None,
    heap_rebuild_factor: float = 3.0,
    output_dir: str | os.PathLike | None = None,
) -> tuple[dict[int, bytes], list[Pair]]:
    start_time = time.perf_counter()
    phase_durations: dict[str, float] = {}

    def record_phase(name: str, phase_start: float) -> None:
        phase_durations[name] = time.perf_counter() - phase_start

    phase_start = time.perf_counter()
    resolved_num_workers = _resolve_num_workers(num_workers)
    input_file_bytes = os.path.getsize(input_path)

    id_to_bytes: dict[int, bytes] = {i: BYTE_TOKENS[i] for i in range(256)}
    vocab_values = set(id_to_bytes.values())
    for special_token in special_tokens:
        special_bytes = special_token.encode("utf-8")
        if special_bytes not in vocab_values:
            id_to_bytes[len(id_to_bytes)] = special_bytes
            vocab_values.add(special_bytes)
    record_phase("vocab_setup", phase_start)

    phase_start = time.perf_counter()
    pretoken_counts = _pretoken_counts(input_path, special_tokens, resolved_num_workers, chunk_bytes)
    record_phase("pretoken_counting", phase_start)
    unique_pretoken_count = len(pretoken_counts)
    total_pretoken_count = sum(pretoken_counts.values())

    phase_start = time.perf_counter()
    word_counts: list[int] = []
    words: list[list[int]] = []
    for pretoken, count in pretoken_counts.items():
        word_counts.append(count)
        words.append(list(pretoken))
    record_phase("word_materialization", phase_start)

    phase_start = time.perf_counter()
    pair_counts, pair_to_word_ids = _build_initial_pair_state(words, word_counts, resolved_num_workers)
    record_phase("initial_pair_state", phase_start)
    initial_pair_count = len(pair_counts)

    heap: list[tuple[int, _ReverseBytesPair, TokenPair]] = []

    def push_pair(pair: TokenPair) -> None:
        count = pair_counts.get(pair, 0)
        if count > 0:
            pair_bytes = (id_to_bytes[pair[0]], id_to_bytes[pair[1]])
            heapq.heappush(heap, (-count, _ReverseBytesPair(pair_bytes), pair))

    def rebuild_heap() -> None:
        heap.clear()
        for pair in pair_counts:
            push_pair(pair)

    phase_start = time.perf_counter()
    rebuild_heap()
    record_phase("initial_heap_build", phase_start)
    initial_heap_size = len(heap)

    def pop_best_pair() -> TokenPair | None:
        while heap:
            neg_count, _, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count > 0 and count == -neg_count:
                return pair
        return None

    heap_rebuild_count = 0
    heap_rebuild_seconds = 0.0

    def maybe_rebuild_heap() -> None:
        nonlocal heap_rebuild_count, heap_rebuild_seconds
        if heap_rebuild_factor <= 0 or not pair_counts:
            return
        if len(heap) > heap_rebuild_factor * len(pair_counts):
            rebuild_start = time.perf_counter()
            rebuild_heap()
            heap_rebuild_seconds += time.perf_counter() - rebuild_start
            heap_rebuild_count += 1

    merges: list[Pair] = []
    merge_loop_start = time.perf_counter()
    merge_pop_best_pair_seconds = 0.0
    merge_word_update_seconds = 0.0
    merge_heap_push_seconds = 0.0
    while len(id_to_bytes) < vocab_size:
        pop_start = time.perf_counter()
        best_pair = pop_best_pair()
        merge_pop_best_pair_seconds += time.perf_counter() - pop_start
        if best_pair is None:
            break

        update_start = time.perf_counter()
        merged_token = id_to_bytes[best_pair[0]] + id_to_bytes[best_pair[1]]
        merged_token_id = len(id_to_bytes)
        merges.append((id_to_bytes[best_pair[0]], id_to_bytes[best_pair[1]]))
        id_to_bytes[merged_token_id] = merged_token

        affected_word_ids = list(pair_to_word_ids.get(best_pair, ()))
        changed_pairs: set[TokenPair] = set()
        for word_id in affected_word_ids:
            word_count = word_counts[word_id]
            old_word = words[word_id]
            old_pairs = _word_pair_frequencies(old_word)

            for pair, frequency in old_pairs.items():
                next_count = pair_counts[pair] - frequency * word_count
                if next_count > 0:
                    pair_counts[pair] = next_count
                else:
                    del pair_counts[pair]

                word_ids = pair_to_word_ids.get(pair)
                if word_ids is not None:
                    word_ids.discard(word_id)
                    if not word_ids:
                        del pair_to_word_ids[pair]
                changed_pairs.add(pair)

            new_word = _merge_word(old_word, best_pair, merged_token_id)
            words[word_id] = new_word
            new_pairs = _word_pair_frequencies(new_word)

            for pair, frequency in new_pairs.items():
                pair_counts[pair] = pair_counts.get(pair, 0) + frequency * word_count
                pair_to_word_ids.setdefault(pair, set()).add(word_id)
                changed_pairs.add(pair)

        merge_word_update_seconds += time.perf_counter() - update_start

        heap_push_start = time.perf_counter()
        for pair in changed_pairs:
            push_pair(pair)
        merge_heap_push_seconds += time.perf_counter() - heap_push_start
        maybe_rebuild_heap()

    record_phase("merge_loop", merge_loop_start)
    merge_loop_subphase_durations = {
        "pop_best_pair": merge_pop_best_pair_seconds,
        "word_rewrite_and_pair_update": merge_word_update_seconds,
        "changed_pair_heap_push": merge_heap_push_seconds,
        "heap_rebuild": heap_rebuild_seconds,
    }

    phase_start = time.perf_counter()
    resolved_output_dir = _write_training_artifacts(id_to_bytes, merges, output_dir, input_path, vocab_size)
    record_phase("artifact_writing", phase_start)
    phase_durations["total_training"] = time.perf_counter() - start_time

    metadata = {
        "format": "cs336_basics.enhanced_bpe.metadata.v1",
        "input_path": os.fspath(input_path),
        "output_dir": str(resolved_output_dir),
        "requested_vocab_size": vocab_size,
        "vocab_size": len(id_to_bytes),
        "merge_count": len(merges),
        "special_tokens": list(special_tokens),
        "num_workers": resolved_num_workers,
        "chunk_bytes": chunk_bytes,
        "heap_rebuild_factor": heap_rebuild_factor,
        "input_file_bytes": input_file_bytes,
        "unique_pretoken_count": unique_pretoken_count,
        "total_pretoken_count": total_pretoken_count,
        "initial_pair_count": initial_pair_count,
        "final_pair_count": len(pair_counts),
        "initial_heap_size": initial_heap_size,
        "final_heap_size": len(heap),
        "heap_rebuild_count": heap_rebuild_count,
        "phase_durations_seconds": phase_durations,
        "phase_durations_formatted": _format_duration_map(phase_durations),
        "merge_loop_subphase_durations_seconds": merge_loop_subphase_durations,
        "merge_loop_subphase_durations_formatted": _format_duration_map(merge_loop_subphase_durations),
    }
    _write_training_metadata(metadata, resolved_output_dir / _METADATA_FILENAME)

    print(f"Enhanced BPE training completed in {_format_duration(time.perf_counter() - start_time)}.", flush=True)
    return id_to_bytes, merges


train_bpe_enhanced = train_bpe

__all__ = ["Pair", "PAT", "PRETOKEN_RE", "TokenPair", "train_bpe", "train_bpe_enhanced"]


### How the enhanced BPE trainer works

The enhanced trainer keeps the same public behavior as the baseline trainer: it returns `dict[int, bytes]` for the vocabulary and an ordered `list[tuple[bytes, bytes]]` for the merges, treats special tokens as hard boundaries, excludes special-token text from merge statistics, and breaks frequency ties by selecting the lexicographically greatest byte pair.

The first enhancement is boundary-safe multiprocessing for pre-tokenization. The corpus is inspected in binary mode and split into byte ranges whose internal boundaries are moved forward to the first configured special token, usually `<|endoftext|>`. Each worker opens the input file independently, reads only its assigned range, decodes it, splits out special tokens, and runs the GPT-2-style regex pre-tokenizer. Workers return only `Counter[bytes]` objects, and the parent process merges those counters. Small files, missing special-token boundaries, or `num_workers=1` fall back to the single-process path.

The second enhancement is parallel initial pair-state construction. Once global pre-token counts are known, the unique pre-token list can be partitioned across workers. Each worker computes local weighted pair counts and local pair-to-word indexes; the parent process reduces these into the global `pair_counts` and `pair_to_word_ids` tables used by the merge loop.

The third enhancement is lower-overhead merge state. Instead of storing each word as a list of `bytes` tokens, the enhanced trainer stores each word as a list of integer token IDs. The vocabulary dictionary still maps IDs to byte strings, so the final vocabulary and merge list stay assignment-compatible. Heap tie-breaking still uses the underlying byte strings, not integer IDs, so the merge order remains deterministic and equivalent to the baseline.

The fourth enhancement is heap maintenance. Like the baseline trainer, the enhanced trainer uses lazy invalidation: old heap entries are ignored when their stored count no longer matches `pair_counts`. For long runs this can leave many stale heap entries behind, so the enhanced version periodically rebuilds the heap when it grows too large relative to the active pair table. This trades a bounded rebuild cost for better long-run memory and pop-time control.

The merge loop itself is intentionally still sequential. After each merge, only pre-token representations containing the selected pair are rewritten, their old pair contributions are removed, and their new pair contributions are added. Parallelizing this outer loop would risk changing the algorithm, because the next best merge depends on the exact statistics produced by the previous merge.


The fifth enhancement is artifact writing and run metadata. After training, the enhanced trainer still returns `(vocab, merges)`, and also writes `vocab.pkl`, `merges.pkl`, `vocab.json`, `merges.txt`, and `metadata.json` to `output_dir` or to the default `<input_stem>_bpe_<vocab_size>/` directory. The pickle files preserve the exact Python objects, while the JSON and text files are meant for human inspection. `metadata.json` records the requested and final vocabulary sizes, merge count, phase durations, merge-loop subphase durations, and run stats. The trainer also prints the total training duration in minutes and seconds after metadata writing completes.

### BPE Training on TinyStories

In this section, we will perform the full BPE Training on the TinyStories data set. 

Write a shell script that invokes the enhanced BPE trainer on the TinyStories dataset, using a maximum vocabulary size of 10,000. Make sure to add the TinyStories `<|endoftext|>` special token to the vocabulary. The enhanced trainer will serialize the resulting vocabulary and merges to disk for further inspection. 

Questions:

(a) How much time did training take? 

(b) What is the longest token in the vocabulary? Does it make sense?

(c) What part of the tokenizer training process takes the most time?

**Answer.**

The enhanced trainer was run with this shell script:

```sh
#!/usr/bin/env bash
set -euo pipefail

uv run python -u -c 'from cs336_basics.train_bpe_enhanced import train_bpe; train_bpe(input_path="data/TinyStoriesV2-GPT4-train.txt", vocab_size=10000, special_tokens=["<|endoftext|>"], num_workers=8, chunk_bytes=64 * 1024 * 1024, heap_rebuild_factor=3.0, output_dir="data/tinystories_bpe_10000")'
```

(a) Training took **1 min 25.57 sec** total, or about **85.57 seconds**, on `data/TinyStoriesV2-GPT4-train.txt`.

(b) The longest vocabulary entries were tied at **15 bytes**: ` accomplishment`, ` disappointment`, and ` responsibility`. This makes sense: they are frequent whole-word tokens in a children's-story corpus, and the leading space is expected because the GPT-2-style pre-tokenizer keeps the preceding space with many word tokens.

(c) The most expensive phase was **pre-token counting**, which took **1 min 22.05 sec** of the **1 min 25.57 sec** total. The merge loop was much smaller by comparison at **3.24 sec**.

### BPE Training on Open Web Text

In this section, we will perform BPE Training on the OpenWebText data set. 

Write a shell script that invokes the enhanced BPE trainer on the OpenWebText dataset, using a maximum vocabulary size of 32,000. Add the `<|endoftext|>` special token to the vocabulary. The enhanced trainer will serialize the resulting vocabulary and merges to disk for further inspection. 

Questions:

(a) How much time did training take? 

(b) What is the longest token in the vocabulary? Does it make sense?

(c) What part of the tokenizer training process takes the most time?

(d) Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.

**Answer.**

The enhanced trainer was run with this shell script:

```sh
#!/usr/bin/env bash
set -euo pipefail

uv run python -u -c 'from cs336_basics.train_bpe_enhanced import train_bpe; train_bpe(input_path="data/owt_train.txt", vocab_size=32000, special_tokens=["<|endoftext|>"], num_workers=8, chunk_bytes=64 * 1024 * 1024, heap_rebuild_factor=3.0, output_dir="data/owt_bpe_32000")'
```

(a) Training took **27 min 7.51 sec** total on `data/owt_train.txt`.

(b) The longest vocabulary entries were tied at **64 bytes**: a repeated mojibake token, `ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ`, and a run of 64 hyphens, `----------------------------------------------------------------`. This makes sense for OpenWebText: byte-level BPE learns frequent repeated byte patterns, and web text contains both formatting separators such as long hyphen runs and noisy encoding artifacts. The mojibake token is not semantically useful in the way a normal word token is, but it is plausible for a noisy web corpus.

(c) The most expensive phase was the **merge loop**, which took **19 min 8.10 sec** of the **27 min 7.51 sec** total. Pre-token counting was the next largest phase at **7 min 26.36 sec**. Inside the merge loop, the largest costs were word rewriting and pair-count updates (**9 min 22.34 sec**) and pushing changed pairs back onto the heap (**7 min 58.99 sec**).

(d) The TinyStories tokenizer is much more specialized. With a 10,000-token vocabulary trained on `data/TinyStoriesV2-GPT4-train.txt`, its longest tokens were normal story words with leading spaces: ` accomplishment`, ` disappointment`, and ` responsibility`. That matches the simpler, cleaner children's-story domain. The OpenWebText tokenizer was trained with a larger 32,000-token vocabulary on a much larger and more diverse web corpus, so it learns many more domain-general tokens, punctuation and formatting patterns, numbers, casing variants, and encoding artifacts. The training profile also changes: TinyStories spent almost all of its time in pre-token counting, while OpenWebText spent most of its time in the sequential merge loop because it had far more unique pre-tokens (**6,601,892** versus **59,933**) and many more requested merges.

### BPE Tokenizer Encoder/Decoder

We will implement a BPE tokenizer that loads a takes a vocabulary and list of merges that was produced by the BPE training process (`cd336/train_bpe.py`, `cs336/train_bpe_enhanced.py`) and uses them to encode and decode a user-supplied text to/from token IDs.

#### 1. Encoding Text

The process of encoding text by BPE mirrors how we train the BPE vocabulary. There are a few major steps:

Step 1: **Pre-tokenize.** 
We first pre-tokenize the sequence and represent each pre-token as a sequence of UTF-8 bytes, just as we did in BPE training. Subsequently, we will be merging these bytes within each pre-token into vocabulary elements, handling each pre-token independently (no merges across pre-token boundaries).

Step 2: **Apply the merges.** 
We then take the sequence of vocabulary element merges created during BPE training, and apply it to our pre-tokens in the same order of creation.

**Special Tokens**: Your tokenizer should be able to properly handle user-defined special tokens when encoding text (provided when constructing the tokenizer).

**Memory Considerations**: To efficiently tokenize large files (or any other stream of data), we need to break it up into manageable chunks and process each chunk in turn, so that the memory complexity is constant as opposed to linear in the size of the text. In doing so, we need to make sure that a token doesn’t cross chunk boundaries, else we’ll get a different tokenization than the naïve method of tokenizing the entire sequence in-memory.

#### 2. Decoding Text

To decode a sequence of integer token IDs back to raw text, we can simply look up each ID’s corresponding entries in the vocabulary (a byte sequence), concatenate them together, and then decode the bytes to a Unicode string. Note that input IDs are not guaranteed to map to valid Unicode strings (since a user could input any sequence of integer IDs). In the case that the input token IDs do not produce a valid Unicode string, you should replace the malformed bytes with the official Unicode replacement character `U+FFFD`. The `errors` argument of `bytes.decode` controls how Unicode decoding errors are handled, and using `errors='replace'` will automatically replace malformed data with the replacement marker.

#### 3. The coding task

Implement a `Tokenizer` Python class that, given a vocabulary and a list of merges produced by the BPE training process, encodes a user-supplied text into integer IDs and decodes integer IDs into text. 

Your tokenizer should also support user-provided special tokens (appending them to the vocabulary if they aren’t already there). 

Use the following interface:

`def __init__(self, vocab, merges, special_tokens=None)` 
  - Construct a tokenizer from a given vocabulary, list of merges, and (optionally) a list of special tokens. This function should accept the following parameters:
    - `vocab: dict[int, bytes]`
    - `merges: list[tuple[bytes, bytes]]`
    - `special_tokens: list[str] | None = None`

`def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None)` 
  - Class method that constructs and returns a Tokenizer from a serialized vocabulary and list of merges (in the same format that your BPE training code output) and (optionally) a list of special tokens. This method should accept the following additional parameters:
    - `vocab_filepath: str`
    - `merges_filepath: str`
    - `special_tokens: list[str] | None = None`

`def encode(self, text: str) -> list[int]`
  - Encode an input text into a sequence of token IDs.

`def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]` 
  - Given an iterable of strings (e.g., a Python file handle), return a generator that lazily yields token IDs. This is required for memory-efficient tokenization of large files that we cannot directly load into memory.

`def decode(self, ids: list[int]) -> str` 
  - Decode a sequence of token IDs into text.

To test your Tokenizer against our provided tests, you will first need to implement the test adapter at [adapters.get_tokenizer] . Then, run `uv run pytest tests/test_tokenizer.py`. Your implementation should be able to pass all tests.

In [ ]:
from __future__ import annotations

import ast
import json
import pickle
from collections.abc import Iterable, Iterator
from pathlib import Path

import regex as re


PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
PRETOKEN_RE = re.compile(PAT)
BYTE_TOKENS = tuple(bytes([i]) for i in range(256))

Pair = tuple[bytes, bytes]
TokenSegment = tuple[int, int, str | None]


class Tokenizer:
    @staticmethod
    def _gpt2_byte_decoder() -> dict[str, int]:
        byte_values = list(range(ord("!"), ord("~") + 1))
        byte_values += list(range(ord("¡"), ord("¬") + 1))
        byte_values += list(range(ord("®"), ord("ÿ") + 1))
        code_points = byte_values[:]
        next_shifted = 0
        for byte_value in range(256):
            if byte_value not in byte_values:
                byte_values.append(byte_value)
                code_points.append(256 + next_shifted)
                next_shifted += 1
        return {chr(code_point): byte_value for byte_value, code_point in zip(byte_values, code_points)}

    @staticmethod
    def _decode_gpt2_token(token: str, byte_decoder: dict[str, int]) -> bytes:
        return bytes(byte_decoder[character] for character in token)

    @classmethod
    def _load_vocab(cls, vocab_filepath: str) -> dict[int, bytes]:
        path = Path(vocab_filepath)
        if path.suffix == ".pkl":
            with path.open("rb") as f:
                return pickle.load(f)

        with path.open(encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict) and isinstance(data.get("tokens"), list):
            vocab: dict[int, bytes] = {}
            for token in data["tokens"]:
                vocab[int(token["id"])] = bytes(token["byte_values"])
            return vocab

        if isinstance(data, dict) and all(isinstance(value, int) for value in data.values()):
            byte_decoder = cls._gpt2_byte_decoder()
            return {token_id: cls._decode_gpt2_token(token, byte_decoder) for token, token_id in data.items()}

        if isinstance(data, dict):
            return {int(token_id): bytes(token_bytes) for token_id, token_bytes in data.items()}

        raise ValueError(f"Unsupported vocabulary file format: {vocab_filepath}")

    @classmethod
    def _load_merges(cls, merges_filepath: str) -> list[Pair]:
        path = Path(merges_filepath)
        if path.suffix == ".pkl":
            with path.open("rb") as f:
                return pickle.load(f)

        byte_decoder = cls._gpt2_byte_decoder()
        merges: list[Pair] = []
        with path.open(encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line or line.startswith("#"):
                    continue

                parts = line.split("\t")
                if len(parts) >= 3 and parts[0].isdigit():
                    merges.append((ast.literal_eval(parts[1]), ast.literal_eval(parts[2])))
                    continue

                parts = line.split(" ")
                if len(parts) == 2:
                    merges.append(
                        (
                            cls._decode_gpt2_token(parts[0], byte_decoder),
                            cls._decode_gpt2_token(parts[1], byte_decoder),
                        )
                    )

        return merges

    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[Pair],
        special_tokens: list[str] | None = None,
    ) -> None:
        self.vocab = dict(vocab)
        self.token_to_id = {token: token_id for token_id, token in self.vocab.items()}

        self.special_tokens = list(special_tokens) if special_tokens is not None else []
        self.special_token_ids: dict[str, int] = {}
        next_token_id = max(self.vocab, default=-1) + 1
        for special_token in self.special_tokens:
            special_bytes = special_token.encode("utf-8")
            token_id = self.token_to_id.get(special_bytes)
            if token_id is None:
                token_id = next_token_id
                next_token_id += 1
                self.vocab[token_id] = special_bytes
                self.token_to_id[special_bytes] = token_id
            self.special_token_ids[special_token] = token_id

        self.merge_ranks = {pair: rank for rank, pair in enumerate(merges)}
        self.byte_token_ids = tuple(self.token_to_id[token] for token in BYTE_TOKENS)
        self.merge_ranks_by_id: dict[tuple[int, int], int] = {}
        self.merge_output_by_pair_id: dict[tuple[int, int], int] = {}
        for pair, rank in self.merge_ranks.items():
            left, right = pair
            pair_ids = (self.token_to_id[left], self.token_to_id[right])
            self.merge_ranks_by_id[pair_ids] = rank
            self.merge_output_by_pair_id[pair_ids] = self.token_to_id[left + right]
        self._encode_cache: dict[bytes, tuple[int, ...]] = {}
        self._max_cache_size = 50_000

        self._max_special_token_length = max((len(token) for token in self.special_tokens), default=0)
        if self.special_tokens:
            special_pattern = "|".join(re.escape(token) for token in sorted(self.special_tokens, key=len, reverse=True))
            self._special_re: re.Pattern[str] | None = re.compile(special_pattern)
        else:
            self._special_re = None

    @classmethod
    def from_files(
        cls,
        vocab_filepath: str,
        merges_filepath: str,
        special_tokens: list[str] | None = None,
    ) -> Tokenizer:
        vocab = cls._load_vocab(vocab_filepath)
        merges = cls._load_merges(merges_filepath)
        return cls(vocab, merges, special_tokens)

    def encode(self, text: str) -> list[int]:
        return list(self._encode_text(text))

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        buffer = ""
        for chunk in iterable:
            if not chunk:
                continue
            buffer += chunk
            segments = list(self._token_segments(buffer))
            flush_index = self._stream_flush_index_from_segments(buffer, segments)
            if flush_index > 0:
                yield from self._encode_prefix_with_context(buffer, flush_index, segments)
                buffer = buffer[flush_index:]
        if buffer:
            yield from self._encode_text(buffer)

    def decode(self, ids: list[int]) -> str:
        return b"".join(self.vocab[token_id] for token_id in ids).decode("utf-8", errors="replace")

    def _encode_text(self, text: str) -> Iterator[int]:
        if self._special_re is None:
            yield from self._encode_normal_text(text)
            return

        start = 0
        for match in self._special_re.finditer(text):
            if match.start() > start:
                yield from self._encode_normal_text(text[start : match.start()])
            yield self.special_token_ids[match.group(0)]
            start = match.end()
        if start < len(text):
            yield from self._encode_normal_text(text[start:])

    def _encode_normal_text(self, text: str) -> Iterator[int]:
        for match in PRETOKEN_RE.finditer(text):
            yield from self._encode_pretoken(match.group(0).encode("utf-8"))

    def _encode_prefix_with_context(
        self,
        text: str,
        end_index: int,
        segments: Iterable[TokenSegment] | None = None,
    ) -> Iterator[int]:
        if segments is None:
            segments = self._token_segments(text)

        for start, end, special_token in segments:
            if end <= end_index:
                if special_token is None:
                    yield from self._encode_pretoken(text[start:end].encode("utf-8"))
                else:
                    yield self.special_token_ids[special_token]
            elif start < end_index:
                raise ValueError("stream flush boundary split a token")
            else:
                break

    def _encode_pretoken(self, pretoken: bytes) -> tuple[int, ...]:
        cached = self._encode_cache.get(pretoken)
        if cached is not None:
            return cached

        tokens = [self.byte_token_ids[byte] for byte in pretoken]
        while len(tokens) > 1:
            best_pair: tuple[int, int] | None = None
            best_rank: int | None = None
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.merge_ranks_by_id.get(pair)
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_pair = pair
                    best_rank = rank

            if best_pair is None:
                break

            merged_token = self.merge_output_by_pair_id[best_pair]
            merged_tokens: list[int] = []
            i = 0
            while i < len(tokens):
                if i + 1 < len(tokens) and tokens[i] == best_pair[0] and tokens[i + 1] == best_pair[1]:
                    merged_tokens.append(merged_token)
                    i += 2
                else:
                    merged_tokens.append(tokens[i])
                    i += 1
            tokens = merged_tokens

        token_ids = tuple(tokens)
        if len(self._encode_cache) >= self._max_cache_size:
            self._encode_cache.clear()
        self._encode_cache[pretoken] = token_ids
        return token_ids

    def _stream_flush_index(self, text: str) -> int:
        return self._stream_flush_index_from_segments(text, list(self._token_segments(text)))

    def _stream_flush_index_from_segments(self, text: str, segments: list[TokenSegment]) -> int:
        if not segments:
            return 0

        keep_start = segments[-1][0]
        if self._max_special_token_length > 1:
            keep_start = min(keep_start, max(0, len(text) - self._max_special_token_length + 1))

        for start, end, _ in segments:
            if start < keep_start < end:
                return start
        return keep_start

    def _token_spans(self, text: str) -> Iterator[tuple[int, int]]:
        for start, end, _ in self._token_segments(text):
            yield start, end

    def _token_segments(self, text: str) -> Iterator[TokenSegment]:
        if self._special_re is None:
            for match in PRETOKEN_RE.finditer(text):
                yield match.start(), match.end(), None
            return

        start = 0
        for special_match in self._special_re.finditer(text):
            if special_match.start() > start:
                for match in PRETOKEN_RE.finditer(text, start, special_match.start()):
                    yield match.start(), match.end(), None
            yield special_match.start(), special_match.end(), special_match.group(0)
            start = special_match.end()
        if start < len(text):
            for match in PRETOKEN_RE.finditer(text, start):
                yield match.start(), match.end(), None


### How the BPE tokenizer works

The updated `Tokenizer` is a deterministic byte-level BPE encoder/decoder built around precomputed lookup tables. During initialization it copies the supplied `dict[int, bytes]` vocabulary, builds `token_to_id`, appends any missing special tokens, and compiles a longest-match regular expression for special tokens. It also converts the merge table into integer-token lookup tables: `byte_token_ids` maps each raw byte to its base token ID, `merge_ranks_by_id` maps adjacent token-ID pairs to their BPE rank, and `merge_output_by_pair_id` maps the same pair to the merged token ID. This lets the encoder run the merge loop on small integers instead of repeatedly concatenating and hashing `bytes` objects.

Encoding first separates configured special tokens from ordinary text. Special tokens are emitted directly as their reserved token IDs and are never split. Non-special spans are pre-tokenized with the GPT-2 regex pattern, then each pre-token is UTF-8 encoded to bytes and initialized as the corresponding single-byte token IDs. The BPE merge loop repeatedly scans adjacent token-ID pairs, selects the available pair with the lowest merge rank, and replaces every non-overlapping occurrence of that pair from left to right. The final list of token IDs is cached by the original pre-token bytes, so repeated words, punctuation runs, and whitespace patterns are encoded once and reused.

`encode` is the simple full-string API: it materializes the complete token-ID list for a `str`. `encode_iterable` is the streaming API for large files. It treats incoming iterable chunks as input batches, appends each chunk to a rolling buffer, computes special-token and regex token segments for the current buffer, and flushes only a prefix whose tokenization is stable. The flush logic keeps the final segment plus enough trailing characters to cover a possible split special-token prefix. When a prefix is flushed, `_encode_prefix_with_context` encodes the already-computed token segments from the full buffer instead of re-tokenizing the prefix as if it ended at EOF. This matters for GPT-2 whitespace tokenization: strings such as `\n\nOnce` can tokenize differently if the two newlines are viewed without the following non-whitespace context. Encoding with the full-buffer segments makes `encode_iterable(chunks)` produce the same token IDs as `encode("".join(chunks))` regardless of chunk boundaries.

Batching is therefore implemented as deterministic streaming over input chunks, not as a vectorized batch-of-strings API. The tokenizer reuses segmentation work inside each streamed chunk and reuses BPE results across chunks through the pre-token cache. It does not launch multiprocessing workers internally. That is intentional for this class: a single serial iterator preserves output order, avoids mutable-cache races, and makes chunk-boundary behavior easy to reason about. If parallel tokenization is needed at a higher level, the safe approach is to split the corpus at known independent boundaries, run separate `Tokenizer` instances in worker processes, and concatenate each worker result in original corpus order.

The algorithm is efficient for a pure-Python assignment implementation because the regex is compiled once, all BPE rank/output lookups are dictionary lookups on integer pairs, repeated pre-tokens hit the cache, and `encode_iterable` avoids loading an entire corpus into memory. The main remaining cost is the merge loop inside a single pre-token: it still rescans adjacent pairs after each merge, so very long individual pre-tokens can be slower than an implementation with a heap or linked-list pair index. In normal GPT-2-style pre-tokenization, most pre-tokens are short, so the integer tables, cache, and streaming buffer provide most of the practical speedup while preserving deterministic output.


### BPE Tokenizer Experiments

#### Experiment 1:

Sample 10 documents from TinyStories and OpenWebText (housed under `data/`). Save these samples to disk under the folder `bpe_samples/` for future reference. This is mainly for audit purposes. 

Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively), encode these sampled documents into integer IDs. Save the integer IDs to the same folder `bpe_samples/`.

Answer the following question: What is each tokenizer’s compression ratio (bytes/token)?

**Experiment 1 answer:** I sampled 10 documents from each training corpus with a deterministic reservoir sampler using seed `336`. The sampled text files are saved under `bpe_samples/tinystories/` and `bpe_samples/openwebtext/`, and the corresponding encoded integer-ID JSON files are saved under `bpe_samples/ids/`.

The TinyStories tokenizer on the TinyStories samples encoded 7,823 bytes into 1,891 tokens, for a compression ratio of **4.137 bytes/token**.

The OpenWebText tokenizer on the OpenWebText samples encoded 93,516 bytes into 24,778 tokens, for a compression ratio of **3.774 bytes/token**.

#### Experiment 2:

Tokenize the OpenWebText samples with the TinyStories tokenizer.

Answer the following questions: 

(a) Compare the compression ratio and qualitatively describe what happens.

(b) Estimate the throughput of your tokenizer (e.g., in bytes/second). How long would it take to tokenize the Pile dataset (825GB of text)?

**Experiment 2 answer:** Tokenizing the same OpenWebText samples with the TinyStories tokenizer encoded 93,516 bytes into 33,213 tokens, for a compression ratio of **2.816 bytes/token**. This is worse than the OpenWebText tokenizer's **3.774 bytes/token** on the same samples. Qualitatively, this happens because the TinyStories tokenizer was trained on a narrower children's-story distribution and has a smaller 10K vocabulary, so it lacks many web-domain merges. OpenWebText text is therefore split into more, shorter tokens, especially for web-specific vocabulary, punctuation patterns, casing variants, numbers, and noisy text artifacts.

The measured TinyStories-tokenizer throughput on these OpenWebText samples was **2,878,179 bytes/second**. At that rate, tokenizing an 825 GB decimal Pile-sized corpus would take about **3 days, 7 hours, 37 minutes, 19.6 seconds**. If interpreting 825 GB as 825 GiB, the estimate is about **3 days, 13 hours, 29 minutes, 36.9 seconds**.

#### Experiment 3:

Using your TinyStories and OpenWebText tokenizers, the task is to encode the respective training and development datasets (in the `data/` folder) into a sequence of integer token IDs. 

**It is required that we serialize the token IDs as a NumPy array of datatype `uint16`.** 

We need to be able to save the outputs to disk in an appropriate format for use later on in training language models. Save the outputs under the `data/` folder which is covered by `.gitignore`: this will avoid commiting large output files to the remote Github repository.

**Experiment 3 resource note:** This experiment is expected to be resource intensive because it asks us to tokenize the full training and development corpora, not just small samples. The full local files are large: `data/TinyStoriesV2-GPT4-train.txt` is about 2.23 GB, `data/TinyStoriesV2-GPT4-valid.txt` is about 22.5 MB, `data/owt_train.txt` is about 11.92 GB, and `data/owt_valid.txt` is about 290 MB. Tokenization must scan all of those bytes and run the Python BPE merge procedure over every pre-token.

The output artifacts are also large. A `uint16` NumPy array uses 2 bytes per token, so the TinyStories training split alone is about 1.08 GB for its 541,229,347 tokens, and the TinyStories validation split is about 10.9 MB for 5,465,883 tokens. Based on the OpenWebText sample compression ratio of about 3.774 bytes/token, the OpenWebText training split is expected to contain roughly 3.16 billion tokens, making just its `uint16` array about 6.3 GB; the OpenWebText validation split is expected to be roughly 76.8 million tokens, or about 154 MB as `uint16`.

Answer the following questions: 

(a) Why is `uint16` an appropriate choice?

(b) What would be efficient and appropriate formats for saving the output to disk?

**Experiment 3 answer (a):** `uint16` is appropriate because the tokenizer vocabulary sizes used here are far below the maximum representable value of an unsigned 16-bit integer. `uint16` can store token IDs from 0 through 65,535, which comfortably covers the TinyStories 10K tokenizer and the OpenWebText 32K tokenizer, including the `<|endoftext|>` special token. It is also much more space-efficient than `uint32`, `int32`, or the `int64` dtype often used by PyTorch tensors: the on-disk corpus costs exactly 2 bytes per token. During training, sampled batches can still be cast to `torch.long` before being passed to the embedding layer, so the compact storage dtype does not restrict model computation.

**Experiment 3 answer (b):** The best primary format is one flat, contiguous NumPy `.npy` file per corpus split, with dtype `np.uint16`, saved under `data/`. For example: `data/tinystories_train_ids.npy`, `data/tinystories_valid_ids.npy`, `data/owt_train_ids.npy`, and `data/owt_valid_ids.npy`. A `.npy` file preserves the array dtype, shape, byte order, and metadata while remaining compact and simple to load.

This format is also appropriate for later LLM training because the training data loader will repeatedly sample random contiguous windows from a large 1D token array. The `.npy` files can be opened with memory mapping, e.g. `np.load("data/owt_train_ids.npy", mmap_mode="r")`, so training can read only the pages needed for each batch instead of loading a multi-GB token array fully into RAM.

If producing the full arrays in one pass is inconvenient, an implementation can write temporary `uint16` `.npy` shards and concatenate them into a final flat `.npy`, or use `numpy.lib.format.open_memmap` to create and fill the final `.npy` directly when the final token count is known. JSON, CSV, plain text, and pickle are poor full-corpus formats here because they are larger, slower, or less transparent for numeric array access. Compressed `.npz` is also not ideal for training because it interferes with efficient random access and memory mapping. A raw `uint16` `.bin` file plus a JSON metadata sidecar would be efficient, but since this experiment requires serializing as a NumPy array, `.npy` is the cleanest and most assignment-compatible choice.

